# Performance Analysis: PN Solver Scaling

This notebook profiles how solve time scales with PN order N, and separates
out where the cost actually goes (matrix assembly, the generalized
eigenproblem, the Marshak linear solve). See `docs/performance.md` for a
written summary.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

from milne_pn.algorithms.pn_system import PNSystem
from milne_pn.algorithms.milne_solver import MilneSolver
from milne_pn.benchmarks.benchmark import timing_benchmark

%matplotlib inline

## 1. End-to-end solve time vs N

In [ ]:
orders = list(range(5, 121, 10))
timing = timing_benchmark(orders, repeats=3)

fig, ax = plt.subplots(figsize=(6.5,4.5))
ax.errorbar(timing["N"], timing["mean_time_s"]*1000, yerr=timing["std_time_s"]*1000,
            fmt="o-", capsize=3)
ax.set_xlabel("PN order N")
ax.set_ylabel("wall time [ms]")
ax.set_title("MilneSolver(N) construction time")
ax.grid(alpha=0.3)
plt.show()

## 2. Where does the time go?

Three dense linear-algebra stages dominate, each roughly $O(n^3)$ in the
matrix size $n=N+1$:

1. `np.linalg.solve(A, D)` to form $M = A^{-1}D$
2. `np.linalg.eig(M)` for the full eigendecomposition
3. `np.linalg.solve(Mtx, rhs)` for the Marshak closure (much smaller: only
   $(N{+}1)/2$ unknowns)

Since (1) and (2) operate on the full $(N{+}1)\times(N{+}1)$ system while (3)
is roughly half that size, (1)-(2) should dominate at large N.

In [ ]:
def timed_stages(N):
    pn = PNSystem(N)
    t0 = time.perf_counter()
    M = np.linalg.solve(pn.A, pn.D)
    t1 = time.perf_counter()
    w, V = np.linalg.eig(M)
    t2 = time.perf_counter()
    return {"assembly_to_M": t1 - t0, "eig": t2 - t1}

orders = [11, 31, 61, 101, 151]  # must be odd
stage_times = {k: [] for k in ("assembly_to_M", "eig")}
for Nord in orders:
    s = timed_stages(Nord)
    for k in stage_times: stage_times[k].append(s[k])

fig, ax = plt.subplots(figsize=(6.5,4.5))
for k, vals in stage_times.items():
    ax.plot(orders, np.array(vals)*1000, "o-", label=k)
ax.set_xlabel("PN order N"); ax.set_ylabel("time [ms]")
ax.legend(); ax.grid(alpha=0.3)
ax.set_title("Cost breakdown: solve(A,D) vs eig(M)")
plt.show()

## 3. Empirical scaling exponent

In [ ]:
# fit log(time) ~ p * log(N) to estimate the effective scaling exponent
logN = np.log(timing["N"].astype(float))
logT = np.log(timing["mean_time_s"])
p, c = np.polyfit(logN, logT, 1)
print(f"empirical scaling: time ~ N^{p:.2f}  (dense O(n^3) linear algebra predicts ~3)")

## 4. Takeaway

Solve time is small in absolute terms (well under a second even at N~150 on
a modern machine). The *asymptotic* cost of `np.linalg.solve` and
`np.linalg.eig` on an $n\times n$ dense matrix is $O(n^3)$, but the
empirical exponent measured above is typically well below 3 at these
matrix sizes ($n\le 150$) -- fixed overhead and BLAS/LAPACK's own
sub-cubic practical behavior at moderate $n$ both flatten the curve before
the cubic regime fully takes over. In any case, this is comfortably fast
enough to run the full convergence study (N up to ~80-100) needed to see
$z_0$ approach Case's 0.7104 to several digits -- see
`notebooks/algorithm_analysis.ipynb` and `docs/performance.md`.

This is also why the GUI's convergence sweep runs on a background thread
purely for UI responsiveness (so the window keeps redrawing/stays
interactive), not because the computation itself is slow -- at these N,
it plainly isn't.